In [2]:
import pandas as pd
from pathlib import Path

In [3]:
data_path = Path(
    "/mnt/c/Users/vetts/Downloads/MimicIII/mimic-iii-clinical-database-1.4"
)

In [4]:
outputevents_sample = pd.read_csv(
    data_path / "OUTPUTEVENTS.csv" / "OUTPUTEVENTS.csv",
    nrows=5
)

outputevents_sample


,ROW_ID,SUBJECT_ID,HADM_ID,ICUSTAY_ID,CHARTTIME,ITEMID,VALUE,VALUEUOM,STORETIME,CGID,STOPPED,NEWBOTTLE,ISERROR
0,344,21219,177991,225765,2142-09-08 10:00:00,40055,200,ml,2142-09-08 12:08:00,17269,NaN,NaN,NaN
1,345,21219,177991,225765,2142-09-08 12:00:00,40055,200,ml,2142-09-08 12:08:00,17269,NaN,NaN,NaN
2,346,21219,177991,225765,2142-09-08 13:00:00,40055,120,ml,2142-09-08 13:39:00,17269,NaN,NaN,NaN
3,347,21219,177991,225765,2142-09-08 14:00:00,40055,100,ml,2142-09-08 16:17:00,17269,NaN,NaN,NaN
4,348,21219,177991,225765,2142-09-08 16:00:00,40055,200,ml,2142-09-08 16:17:00,17269,NaN,NaN,NaN


In [5]:
d_items_df = pd.read_csv(
    data_path / "D_ITEMS.csv" / "D_ITEMS.csv"
)

outputevents_labeled = outputevents_sample.merge(
    d_items_df[["ITEMID", "LABEL"]],
    on="ITEMID",
    how="left"
)

outputevents_labeled

,ROW_ID,SUBJECT_ID,HADM_ID,ICUSTAY_ID,CHARTTIME,ITEMID,VALUE,VALUEUOM,STORETIME,CGID,STOPPED,NEWBOTTLE,ISERROR,LABEL
0,344,21219,177991,225765,2142-09-08 10:00:00,40055,200,ml,2142-09-08 12:08:00,17269,NaN,NaN,NaN,Urine Out Foley
1,345,21219,177991,225765,2142-09-08 12:00:00,40055,200,ml,2142-09-08 12:08:00,17269,NaN,NaN,NaN,Urine Out Foley
2,346,21219,177991,225765,2142-09-08 13:00:00,40055,120,ml,2142-09-08 13:39:00,17269,NaN,NaN,NaN,Urine Out Foley
3,347,21219,177991,225765,2142-09-08 14:00:00,40055,100,ml,2142-09-08 16:17:00,17269,NaN,NaN,NaN,Urine Out Foley
4,348,21219,177991,225765,2142-09-08 16:00:00,40055,200,ml,2142-09-08 16:17:00,17269,NaN,NaN,NaN,Urine Out Foley


In [8]:
outputevents_df = pd.read_csv(
    data_path / "OUTPUTEVENTS.csv" / "OUTPUTEVENTS.csv",
    usecols=["ICUSTAY_ID", "ITEMID", "CHARTTIME", "VALUE", "VALUEUOM"],
    parse_dates=["CHARTTIME"]
)

In [9]:
item_icu_counts = (
    outputevents_df
    .groupby("ITEMID")["ICUSTAY_ID"]
    .nunique()
    .sort_values(ascending=False)
)

item_icu_counts.head(20)

ITEMID
40055     23750
226559    19795
40054     10911
226560     8015
40069      7547
40059      5302
226627     5085
40076      4665
40060      4528
226633     4348
40065      4033
43175      3587
40052      3477
226588     3429
40064      3269
40061      3261
226626     2979
226576     2967
227510     2709
40067      2229
Name: ICUSTAY_ID, dtype: int64

In [10]:
item_counts_df = item_icu_counts.reset_index(name="icu_count")

item_counts_labeled = item_counts_df.merge(
    d_items_df[["ITEMID", "LABEL"]],
    on="ITEMID",
    how="left"
)

item_counts_labeled.head(20)

,ITEMID,icu_count,LABEL
0,40055,23750,Urine Out Foley
1,226559,19795,Foley
2,40054,10911,Stool Out Stool
3,226560,8015,Void
4,40069,7547,Urine Out Void
5,40059,5302,Gastric Oral Gastric
6,226627,5085,OR Urine
7,40076,4665,Chest Tubes CTICU CT 1
8,40060,4528,Pre-Admission Output Pre-Admission Output
9,226633,4348,Pre-Admission


In [11]:
urine_labels_df = item_counts_labeled[
    item_counts_labeled["LABEL"].str.contains("Urine|Foley|Void", case=False, na=False)
]

urine_labels_df

,ITEMID,icu_count,LABEL
0,40055,23750,Urine Out Foley
1,226559,19795,Foley
3,226560,8015,Void
4,40069,7547,Urine Out Void
6,226627,5085,OR Urine
...,...,...,...
1142,44080,1,EW Urine
1147,44103,1,ER urine out
1148,44132,1,Procedure urine out
1153,44237,0,E.R. urine out


In [12]:
pd.set_option("display.max_rows", None)

urine_labels_df.sort_values(
    "icu_count",
    ascending=False
)

,ITEMID,icu_count,LABEL
0,40055,23750,Urine Out Foley
1,226559,19795,Foley
3,226560,8015,Void
4,40069,7547,Urine Out Void
6,226627,5085,OR Urine
10,40065,4033,OR Out PACU Urine
11,43175,3587,Urine .
15,40061,3261,OR Out OR Urine
30,40094,907,Urine Out Condom Cath
37,40288,589,PACU Out PACU Urine


In [13]:
urine_itemids = [
    # CareVue
    40055, 43175, 40069, 40094, 40715,
    40473, 40085, 40057, 40056, 40405,
    40428, 40096, 40651,

    # MetaVision
    226559, 226560, 226561, 226584, 226563,
    226564, 226565, 226567, 226557, 226558
]

In [14]:
urine_output_df = outputevents_df[
    outputevents_df["ITEMID"].isin(urine_itemids)
].copy()

urine_output_df.head()

,ICUSTAY_ID,CHARTTIME,ITEMID,VALUE,VALUEUOM
0,225765.0,2142-09-08 10:00:00,40055,200.0,ml
1,225765.0,2142-09-08 12:00:00,40055,200.0,ml
2,225765.0,2142-09-08 13:00:00,40055,120.0,ml
3,225765.0,2142-09-08 14:00:00,40055,100.0,ml
4,225765.0,2142-09-08 16:00:00,40055,200.0,ml


In [15]:
cohort_df = pd.read_parquet(
    "../data/processed/adult_icu_cohort_first24h.parquet"
)

icu_times_df = cohort_df[["ICUSTAY_ID", "INTIME"]]

In [16]:
urine_first24h_df = urine_output_df.merge(
    icu_times_df,
    on="ICUSTAY_ID",
    how="inner"
)

In [17]:
urine_first24h_df = urine_first24h_df[
    (urine_first24h_df["CHARTTIME"] >= urine_first24h_df["INTIME"]) &
    (urine_first24h_df["CHARTTIME"] < urine_first24h_df["INTIME"] + pd.Timedelta(hours=24))
]

In [18]:
urine_features_df = (
    urine_first24h_df
    .groupby("ICUSTAY_ID")["VALUE"]
    .sum()
    .reset_index(name="urine_output_24h")
)

urine_features_df.head()

,ICUSTAY_ID,urine_output_24h
0,200001.0,250.0
1,200003.0,3652.0
2,200006.0,1955.0
3,200007.0,1295.0
4,200009.0,1570.0


In [19]:
urine_features_df["ICUSTAY_ID"] = urine_features_df["ICUSTAY_ID"].astype(int)

In [20]:
urine_cohort_df = cohort_df[["ICUSTAY_ID"]].merge(
    urine_features_df,
    on="ICUSTAY_ID",
    how="left"
)

urine_missing_count = urine_cohort_df["urine_output_24h"].isna().sum()

urine_missing_percent = (
    urine_cohort_df["urine_output_24h"].isna().mean() * 100
)

print("Total ICU stays:", len(urine_cohort_df))
print("Missing urine output:", urine_missing_count)
print("Missing percent:", urine_missing_percent)

Total ICU stays: 45253
Missing urine output: 2665
Missing percent: 5.889112324044814


## Urine Output Feature Engineering

- Urine-related records were identified from `OUTPUTEVENTS` using relevant `ITEMID` values.
- Only raw urine volume records were kept; rate-based and irrigation-related records were excluded.
- Urine records were matched with ICU admission time (`INTIME`).
- Only measurements within the first 24 hours of each ICU stay were retained.
- Total urine output was calculated for each `ICUSTAY_ID`.
- `ICUSTAY_ID` was converted to integer format.
- Missingness was checked against the full cohort of 45,253 ICU stays.
- Urine output was missing in 2,665 stays (~5.89%).

In [21]:
chest_labels_df = item_counts_labeled[
    item_counts_labeled["LABEL"].str.contains(
        "chest|pleural|mediastinal",
        case=False,
        na=False
    )
]

chest_labels_df.sort_values(
    "icu_count",
    ascending=False
)

,ITEMID,icu_count,LABEL
7,40076,4665,Chest Tubes CTICU CT 1
13,226588,3429,Chest Tube #1
27,40049,1016,Chest Tubes Right Pleural 1
28,40048,988,Chest Tubes Left Pleural 1
42,226589,557,Chest Tube #2
57,226593,284,R Pleural #1
59,226590,279,L Pleural #1
60,41707,276,Chest Tubes CTICU CT 2
62,40091,270,Chest Tubes Mediastinal
65,226592,250,Mediastinal


In [22]:
chest_tube_itemids = [
    40076, 226588,
    40049, 40048,
    226589,
    226593, 226590,
    41707,
    40091, 226592,
    40050, 40090,
    40084,
    226591, 226595,
    43177
]

In [23]:
chest_tube_df = outputevents_df[
    outputevents_df["ITEMID"].isin(chest_tube_itemids)
].copy()

chest_tube_df.head()

,ICUSTAY_ID,CHARTTIME,ITEMID,VALUE,VALUEUOM
2150,285436.0,2188-07-24 06:00:00,40050,110.0,ml
10048,279993.0,2183-09-23 06:00:00,40049,20.0,ml
10225,279993.0,2183-09-23 17:00:00,40049,32.0,ml
10226,279993.0,2183-09-23 20:00:00,40049,17.0,ml
10227,279993.0,2183-09-24 06:00:00,40049,20.0,ml


In [24]:
chest_first24h_df = chest_tube_df.merge(
    icu_times_df,
    on="ICUSTAY_ID",
    how="inner"
)

chest_first24h_df = chest_first24h_df[
    (chest_first24h_df["CHARTTIME"] >= chest_first24h_df["INTIME"]) &
    (chest_first24h_df["CHARTTIME"] < chest_first24h_df["INTIME"] + pd.Timedelta(hours=24))
]

chest_features_df = (
    chest_first24h_df
    .groupby("ICUSTAY_ID")["VALUE"]
    .sum()
    .reset_index(name="chest_tube_output_24h")
)

chest_features_df["ICUSTAY_ID"] = chest_features_df["ICUSTAY_ID"].astype(int)

chest_features_df.head()

,ICUSTAY_ID,chest_tube_output_24h
0,200009,2158.0
1,200025,940.0
2,200062,870.0
3,200063,320.0
4,200069,580.0


In [25]:
chest_cohort_df = cohort_df[["ICUSTAY_ID"]].merge(
    chest_features_df,
    on="ICUSTAY_ID",
    how="left"
)

chest_missing_count = chest_cohort_df["chest_tube_output_24h"].isna().sum()

chest_missing_percent = (
    chest_cohort_df["chest_tube_output_24h"].isna().mean() * 100
)

print("Total ICU stays:", len(chest_cohort_df))
print("Missing chest tube output:", chest_missing_count)
print("Missing percent:", chest_missing_percent)

Total ICU stays: 45253
Missing chest tube output: 37147
Missing percent: 82.08737542262391


In [26]:
chest_binary_df = cohort_df[["ICUSTAY_ID"]].copy()

chest_binary_df["chest_tube_present"] = (
    chest_binary_df["ICUSTAY_ID"]
    .isin(chest_features_df["ICUSTAY_ID"])
    .astype(int)
)

chest_binary_df.head()

,ICUSTAY_ID,chest_tube_present
0,280836,0
1,206613,0
2,220345,0
3,249196,0
4,210407,0


In [27]:
chest_binary_df["chest_tube_present"].value_counts()

chest_tube_present
0    37147
1     8106
Name: count, dtype: int64

In [28]:
ebl_labels_df = item_counts_labeled[
    item_counts_labeled["LABEL"].str.contains(
        "EBL|blood loss",
        case=False,
        na=False
    )
]

ebl_labels_df.sort_values(
    "icu_count",
    ascending=False
)

,ITEMID,icu_count,LABEL
14,40064,3269,OR Out EBL
16,226626,2979,OR EBL
112,40491,61,PACU Out EBL
121,226629,49,PACU EBL


In [29]:
ebl_itemids = [40064, 226626, 40491, 226629]

In [30]:
ebl_df = outputevents_df[
    outputevents_df["ITEMID"].isin(ebl_itemids)
].copy()

ebl_first24h_df = ebl_df.merge(
    icu_times_df,
    on="ICUSTAY_ID",
    how="inner"
)

ebl_first24h_df = ebl_first24h_df[
    (ebl_first24h_df["CHARTTIME"] >= ebl_first24h_df["INTIME"]) &
    (ebl_first24h_df["CHARTTIME"] < ebl_first24h_df["INTIME"] + pd.Timedelta(hours=24))
]

ebl_features_df = (
    ebl_first24h_df
    .groupby("ICUSTAY_ID")["VALUE"]
    .sum()
    .reset_index(name="ebl_24h")
)

ebl_features_df["ICUSTAY_ID"] = ebl_features_df["ICUSTAY_ID"].astype(int)

In [31]:
ebl_cohort_df = cohort_df[["ICUSTAY_ID"]].merge(
    ebl_features_df,
    on="ICUSTAY_ID",
    how="left"
)

ebl_cohort_df["ebl_present"] = (
    ebl_cohort_df["ebl_24h"].notna().astype(int)
)

ebl_cohort_df.head()

,ICUSTAY_ID,ebl_24h,ebl_present
0,280836,NaN,0
1,206613,NaN,0
2,220345,NaN,0
3,249196,NaN,0
4,210407,NaN,0


In [32]:
ebl_cohort_df["ebl_present"].value_counts()

ebl_present
0    40962
1     4291
Name: count, dtype: int64

In [33]:
ebl_cohort_df.loc[
    ebl_cohort_df["ebl_present"] == 1,
    "ebl_24h"
].describe()

count     4291.000000
mean       945.017129
std       2242.186254
min          0.000000
25%        100.000000
50%        300.000000
75%        800.000000
max      50000.000000
Name: ebl_24h, dtype: float64

In [34]:
ebl_cohort_df.loc[
    ebl_cohort_df["ebl_present"] == 1,
    "ebl_24h"
].quantile([0.95, 0.99, 0.995, 0.999])

0.950     4000.0
0.990    10000.0
0.995    13685.0
0.999    25000.0
Name: ebl_24h, dtype: float64

In [35]:
output_features_df = chest_binary_df.merge(
    urine_features_df,
    on="ICUSTAY_ID",
    how="left"
)

output_features_df = output_features_df.merge(
    ebl_cohort_df[["ICUSTAY_ID", "ebl_present", "ebl_24h"]],
    on="ICUSTAY_ID",
    how="left"
)

output_features_df = output_features_df[
    [
        "ICUSTAY_ID",
        "urine_output_24h",
        "chest_tube_present",
        "ebl_present",
        "ebl_24h"
    ]
]

output_features_df.head()

,ICUSTAY_ID,urine_output_24h,chest_tube_present,ebl_present,ebl_24h
0,280836,216.0,0,0,NaN
1,206613,3850.0,0,0,NaN
2,220345,2870.0,0,0,NaN
3,249196,1195.0,0,0,NaN
4,210407,3215.0,0,0,NaN


In [36]:
output_features_df.to_parquet(
    "../data/processed/output_features_first24h.parquet",
    index=False
)

## Additional Output Features

- Chest tube-related output records were identified from `OUTPUTEVENTS`.
- Chest tube output was converted into a binary feature: `chest_tube_present`.
- `chest_tube_present = 1` indicates a chest tube record within the first 24 hours, otherwise `0`.
- EBL (Estimated Blood Loss) records were identified from OR and PACU output items.
- Two EBL features were created:
  - `ebl_present`: whether an EBL record exists in the first 24 hours.
  - `ebl_24h`: total estimated blood loss in the first 24 hours.
- EBL showed strong outliers, but they were kept unchanged for later model-stage evaluation.
- Urine, chest tube, and EBL features were combined into a single output feature table.
- The final table was saved as `output_features_first24h.parquet`.